In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

import urllib.request

urllib.request.urlretrieve(data_url, "input.txt")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [2]:
with open ('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset", len(text))

length of dataset 1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



Next we extraxt the vocab of the text

In [5]:
chars = sorted(list(set(text)))

vocab_len = len(chars)

print("Characters: ",''.join(chars))

print("vocanbulary size: ",vocab_len)

Characters:  
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocanbulary size:  65


### Tokenize the input text

Tokenize : Convert raw text into some meaning full words according to some vabualry

Now here we are building encoder and decoder to map the characters to string 

- Google uses Sentence Piece uses, subword tokenization
- OpenAI has library tiktoken that has vocan range of 50257
so instead of only 65 possible combinations we have 50257 possible combinations

In [6]:
stoi = {ch:i for i, ch in enumerate(chars)}

itos = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[c] for c in l])
decode_split = lambda l: [itos[c] for c in l]



strg = "Hi I am Malay Jain"
print(encode(strg))
print(decode_split(encode(strg)))
print(decode(encode(strg)))


[20, 47, 1, 21, 1, 39, 51, 1, 25, 39, 50, 39, 63, 1, 22, 39, 47, 52]
['H', 'i', ' ', 'I', ' ', 'a', 'm', ' ', 'M', 'a', 'l', 'a', 'y', ' ', 'J', 'a', 'i', 'n']
Hi I am Malay Jain


In [7]:
print(decode([0, 1, 2]))
print(" \' ",decode([0])," \'\' ",decode([1]), " \'\' ",decode([2]), " \' ")


 !
 '  
  ''     ''  !  ' 


in above 0 --> new line character and 1 --> blank space

## Now let's encode the complete input.txt using our tokenizer
using PyTorch Library

In [8]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)

print(data.shape, data.dtype)

print(data[:1000]) # here data is the tokenized tensor of the original input text

c:\Users\TRIO990019\Desktop\Complete-Backup\GPT-mini\venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

### Seperating dataset in training and test data

In [9]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[:n]

print(n)
print(len(data))

1003854
1115394


We train the data using the chunk size, lets say, we put it as a block size.
Here block size means the number of characters we are using

In [10]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

Here we train the transformer for every tensor, here for batch size 8 there become around 8 examples, for each tensor the transformer need to predict the next value, we can usnderstand it from following below

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("Example:",t,"= For input",{context},"output will be: ",{target})

Example: 0 = For input {tensor([18])} output will be:  {tensor(47)}
Example: 1 = For input {tensor([18, 47])} output will be:  {tensor(56)}
Example: 2 = For input {tensor([18, 47, 56])} output will be:  {tensor(57)}
Example: 3 = For input {tensor([18, 47, 56, 57])} output will be:  {tensor(58)}
Example: 4 = For input {tensor([18, 47, 56, 57, 58])} output will be:  {tensor(1)}
Example: 5 = For input {tensor([18, 47, 56, 57, 58,  1])} output will be:  {tensor(15)}
Example: 6 = For input {tensor([18, 47, 56, 57, 58,  1, 15])} output will be:  {tensor(47)}
Example: 7 = For input {tensor([18, 47, 56, 57, 58,  1, 15, 47])} output will be:  {tensor(58)}


Ok now we will perfrom the parallel processing here



In [12]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data # picks the correct dataset based on the split argument

    ix = torch.randint(len(data) - block_size, (batch_size,)) # sets (low,high,size) here low is set 0 by default and high is set, and size is in tuple
    # the above line generates the 4 random starting positions
    print(ix)
    # following takes the 4 random starting positions to genereate the 4 lists of size 8 and stack them one above another.
    x = torch.stack([data[i:i+block_size] for i in ix]) # stacks the multiple generated tensors into a single tensor, so that we can have a batch of inputs

    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

xb, yb = get_batch('train') # ok this function will generate the tensors for batch processing in the batch of 8 each

print('inputs:')
print(xb.shape)
print("xb = ",xb)
print(yb.shape)
print("yb = ",yb)

print("-----")

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"When context is {context} then target is : {target}")
    print("\n")

tensor([ 76049, 234249, 934904, 560986])
inputs:
torch.Size([4, 8])
xb =  tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
yb =  tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
When context is tensor([24]) then target is : 43
When context is tensor([24, 43]) then target is : 58
When context is tensor([24, 43, 58]) then target is : 5
When context is tensor([24, 43, 58,  5]) then target is : 57
When context is tensor([24, 43, 58,  5, 57]) then target is : 1
When context is tensor([24, 43, 58,  5, 57,  1]) then target is : 46
When context is tensor([24, 43, 58,  5, 57,  1, 46]) then target is : 43
When context is tensor([24, 43, 58,  5, 57,  1, 46, 43]) then target is : 39


When context is tensor([44]) then target is : 53
When

In [13]:
print(xb) # our transformer input is here

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


Reference read the Biagram Language Model.....

In [14]:
vocab_size =  vocab_len

# Here first take a break and study the makemore series

Ok so in the biagram table, when the model is applied on the data, the following happens:             

say input is [2,4,6,7,8] for the embedding is of 10x10, the vales reads

Every single integer, refers to the embedding table and plucks out the row corresponding to it's own value, then pytorch arranges them in BxTxC order

These plucked out arranged values are termed as logits which are the scores for the next character in the sequence

Prodcting what comes next on the basis of the individual identity of the single token

In [15]:
import torch
import torch.nn as nn

from torch.nn import functional as F
torch.manual_seed(1337)

class BiagramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token reads off the logits for the next toekn  from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) #(B,T,C) pytorch arranges the data in Batch x Time x Channel --> 4x8x65

        # now to evalute loss

        ## how here the pytorch library expects the multi-dimentional input for the "cross_entropy" in BxCxT format not in BxTxC which our tensor currently is
        # so we reshape our logits

        B, T, C = logits.shape
        logits = logits.view(B*T, C) # so we convert our actuall tensor in a single value and seperate out the channel 
        targets = targets.view(B*T) # .view changes the shape of the tensor, does not change the order of the values and the values

        loss = F.cross_entropy(logits, targets) # this measure the quality of the logits with respect to the targets

        return logits, loss
    

m = BiagramLanguageModel(vocab_size)
logits, loss = m(xb,yb)

print(logits.shape)
print("Predicted scores for each valeus in out 4x8 positions")
print(logits)
print(f"Loss: {loss.item():.4f}")

torch.Size([32, 65])
Predicted scores for each valeus in out 4x8 positions
tensor([[-1.5101, -0.0948,  1.0927,  ..., -0.6126, -0.6597,  0.7624],
        [ 0.3323, -0.0872, -0.7470,  ..., -0.6716, -0.9572, -0.9594],
        [ 0.2475, -0.6349, -1.2909,  ...,  1.3064, -0.2256, -1.8305],
        ...,
        [-2.1910, -0.7574,  1.9656,  ..., -0.3580,  0.8585, -0.6161],
        [ 0.5978, -0.0514, -0.0646,  ..., -1.4649, -2.0555,  1.8275],
        [-0.6787,  0.8662, -1.6433,  ...,  2.3671, -0.7775, -0.2586]],
       grad_fn=<ViewBackward0>)
Loss: 4.8786


ok so for these 65 values
-ln(1/65) = 4.174... should be the loss
but here the loss is 4.876

Input text             
     ↓             
Tokenizer             
     ↓             
Token IDs             
     ↓             
Neural Network (GPT)             
     ↓             
Logits (raw scores)             
     ↓             
Softmax             
     ↓             
Probabilities             
     ↓             
Cross Entropy             
     ↓             
Loss             

**Softmax does NOT convert the input into logits.**

It's actually the other way around.

The complete pipeline is:

```text
Input text
     ↓
Tokenizer
     ↓
Token IDs
     ↓
Neural Network (GPT)
     ↓
Logits (raw scores)
     ↓
Softmax
     ↓
Probabilities
     ↓
Cross Entropy
     ↓
Loss
```

Let's go through an example.

---

## Step 1: Input

Suppose your text is:

```text
h
```

The tokenizer converts it into an ID.

```text
'h' → 15
```

---

## Step 2: Neural Network

The token ID goes through the entire GPT model (embeddings, attention, feed-forward layers, etc.).

The model finally outputs one score for **every possible next token**.

Suppose your vocabulary is:

```text
a
b
c
d
```

The model outputs:

```text
a = 2
b = 5
c = 1
d = 3
```

These are called **logits**.

Notice:

* They are just scores.
* They are **not** probabilities.
* They don't sum to 1.

---

## Step 3: Softmax

Softmax takes these logits:

```text
a = 2
b = 5
c = 1
d = 3
```

and converts them into probabilities:

```text
a = 0.04

b = 0.84

c = 0.02

d = 0.10
```

Now they:

* are between 0 and 1
* sum to exactly 1

---

## Step 4: Cross Entropy

Suppose the correct next character is

```text
b
```

Cross Entropy looks at the probability assigned to **b**.

Here:

```text
b = 84%
```

That's good, so the loss is small.

---

## A simple analogy

Imagine you're judging a singing competition.

The contestants are:

```text
Alice
Bob
Charlie
David
```

The judges give **scores**:

```text
Alice   72

Bob     95

Charlie 68

David   80
```

These scores are like **logits**.

Then Softmax converts them into:

```text
Alice   5%

Bob     80%

Charlie 2%

David   13%
```

Now they become probabilities.

---

### So remember this:

* **Input** → goes into the neural network.
* **Neural network** → produces **logits** (raw scores).
* **Softmax** → converts those logits into probabilities.
* **Cross Entropy** → compares those probabilities with the correct answer and computes the loss.

A common beginner mistake is to think Softmax creates the logits. It doesn't. **The neural network creates the logits, and Softmax simply transforms them into probabilities.**
